# 实验6：昇腾计算语言AscendCL应用

## 一、实验目的
1. 掌握基于昇腾AscendCL（Ascend Compute Language）的编程流程，理解ACL接口在资源管理、模型推理中的作用。  
2. 熟练使用ACL设备管理、内存管理、模型加载与执行等接口，具备ACL应用开发与流程控制能力。  
3. 初步接触昇腾Ascend C（原TIK C++）算子编程，理解自定义算子如何在ACL应用中加载与调用，建立“算子+框架”的全流程概念。

## 二、实验说明

### 2.1 实验背景
昇腾AI处理器提供了一套完整的软件栈，其中**AscendCL**（Ascend Compute Language，简称ACL）是面向应用开发者的C语言API库，负责设备初始化、内存管理、模型加载、推理执行和资源释放等任务。通过ACL编程，用户可以灵活地控制推理流程，实现高性能的AI业务逻辑。  
同时，昇腾还提供了**Ascend C**算子开发语言，用于编写可在NPU上执行的自定义算子。将自定义算子编译为离线模型后，同样可以通过ACL接口加载运行。本实验将ACL基础应用与Ascend C简单实践相结合，帮助同学们建立完整的昇腾计算编程视角。

### 2.2 实验环境准备（**CANNLab在线实验路过这一环节**）
本实验依赖华为昇腾沙箱实验环境，已预装以下组件：
- 操作系统：openEuler / Ubuntu（CANN适配版本）
- 昇腾NPU驱动及固件
- CANN软件包（包含AscendCL、Ascend C编译器、工具链等）
- 代码编辑器（如Vim/VS Code Remote）、GCC编译器、CMake等

登录后，请确认环境可用：
```bash
npu-smi info        # 查看NPU状态
cat /usr/local/Ascend/ascend-toolkit/latest/version.cfg  # 查看CANN版本
```

设置环境变量（可写入 `~/.bashrc`）：
```bash
export ASCEND_HOME=/usr/local/Ascend/ascend-toolkit/latest
export PATH=$ASCEND_HOME/bin:$PATH
export LD_LIBRARY_PATH=$ASCEND_HOME/lib64:$LD_LIBRARY_PATH
```

## 三、实验任务

### 3.1 任务描述
本实验包含两个子任务：
1. **ACL基础推理应用**：使用AscendCL编写一个图像分类推理程序，加载一个预置的ResNet-50离线模型（`resnet50.om`），对一张图像进行推理，输出Top-5分类结果。过程中重点完成ACL初始化、设备内存申请与拷贝、模型加载、数据预处理、推理执行和资源释放。
2. **Ascend C自定义算子集成**（选做/拓展）：使用Ascend C编写一个简单的向量加法算子，编译为离线模型，并在ACL应用中加载运行，体会从算子开发到ACL调用的全流程。

### 3.2 学习目标
- 熟悉ACL标准编程流程：`aclInit` → `aclrtSetDevice` → `aclrtCreateContext` → `aclrtCreateStream` → 资源申请与拷贝 → 模型加载执行 → 资源释放。
- 掌握ACL内存管理接口：`aclrtMalloc`、`aclrtMemcpy`、`acldvppMalloc`（可选）等，区分Host与Device内存。
- 理解模型描述信息获取方式、输入输出Dataset的创建与销毁。
- 能够阅读并修改ACL示例代码，完成基本的推理任务。
- 了解Ascend C算子内核编写、编译及ACL加载流程，为后续深入学习打下基础。

## 四、任务准备

### 4.1 前置知识
- C/C++语言编程基础，熟悉指针、内存管理、文件操作。
- 了解Linux基本命令，能够使用命令行编译、运行程序。
- 理解神经网络推理的基本流程：输入数据预处理、模型计算、输出解析。
- 对昇腾AI处理器的Device/Host概念有初步认识。

### 4.2 实验数据准备
在沙箱的家目录下创建实验目录并获取所需文件：
```bash
mkdir -p /home/developer/experiment/lab_06 /home/developer/experiment/lab_06/src
cd /home/developer/experiment/lab_06
```
本实验提供以下材料（由教师分发或从沙箱公共目录拷贝）：
- 离线模型文件：`resnet50.om`（已通过ATC工具转换，输入为`float32`，形状`1,3,224,224`）
- 测试图片：`dog.jpg`（一张尺寸约为224×224的彩色图片）
- 图片预处理脚本或预处理后的二进制数据：`preprocess.py` 或 `input_dog.bin`
- 分类标签文件：`imagenet_labels.txt`（1000类标签）

将上述文件分别放入 `lab_06/` 目录下。如果需要自行预处理，可使用`preprocess.py`将图片缩放、居中裁剪并转为 CHW 排布的 float32 二进制文件，尺寸为224×224。


## 五、任务实施

#### 步骤1：编写ACL推理主程序
在 `/home/developer/experiment/lab_06/src/` 下创建 `main.c`，按以下框架完成代码（关键函数需自行补全或参照注释调用ACL API）。

**程序整体结构：**
```c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include "acl/acl.h"

#define MODEL_PATH "../resnet50.om"
#define INPUT_BIN_PATH "../input_cat.bin"
#define LABEL_PATH "../imagenet_labels.txt"
#define NUM_CLASSES 1000
#define TOP_K 5

char** load_imagenet_labels(const char *filename, int *num_labels) {
    FILE *file = fopen(filename, "r");
    if (!file) {
        printf("Error: Cannot open labels file '%s'\n", filename);
        *num_labels = 0;
        return NULL;
    }
    
    char **labels = (char**)malloc(1000 * sizeof(char*));
    char line[1024];
    int count = 0;
    
    while (fgets(line, sizeof(line), file) && count < 1000) {
        line[strcspn(line, "\n")] = 0;
        labels[count] = (char*)malloc(strlen(line) + 1);
        strcpy(labels[count], line);
        count++;
    }
    
    fclose(file);
    *num_labels = count;

    return labels;
}

void free_labels(char **labels, int num_labels) {
    for (int i = 0; i < num_labels; i++) {
        free(labels[i]);
    }
    free(labels);
}

void softmax(float *input, float *output, int size) {
    float max = input[0];
    for (int i = 1; i < size; i++) {
        if (input[i] > max) max = input[i];
    }
    
    float sum = 0.0f;
    for (int i = 0; i < size; i++) {
        output[i] = expf(input[i] - max);
        sum += output[i];
    }
    
    for (int i = 0; i < size; i++) {
        output[i] /= sum;
    }
}

void find_top_k(float *probs, int *indices, int k, int size) {
    // Initialize indices array
    for (int i = 0; i < k; i++) {
        indices[i] = -1;
    }
    
    // Find top k
    for (int i = 0; i < size; i++) {
        for (int j = 0; j < k; j++) {
            if (indices[j] == -1 || probs[i] > probs[indices[j]]) {
                // Shift lower values
                for (int l = k - 1; l > j; l--) {
                    indices[l] = indices[l - 1];
                }
                indices[j] = i;
                break;
            }
        }
    }
}

void print_top_k_results(float *probs, int *indices, int k, char **labels) {
    printf("\n=== Top-%d Predictions ===\n", k);
    for (int i = 0; i < k; i++) {
        int idx = indices[i];
        printf("%d. Index: %d, Probability: %.4f, Label: %s\n", 
               i + 1, idx, probs[idx], 
               labels[idx] ? labels[idx] : "Unknown");
    }
}

int main() {
    aclError ret;
    uint32_t deviceId = 0;
    aclrtContext context;
    
    uint32_t modelId;
    aclmdlDesc *modelDesc = NULL;
    aclmdlDataset *inputDataset = NULL, *outputDataset = NULL;
    
    void *inputHostBuf = NULL, *inputDevBuf = NULL;
    void *outputHostBuf = NULL, *outputDevBuf = NULL;
    size_t inputSize, outputSize;

    // TODO
    //

    // 1. ACL初始化
    
    // 2. 设置设备

    // 3. 创建上下文
    
    // 4. 加载模型（从文件）
    
    // 5. 准备输入数据
    
    // 6. 执行推理

    // 7. 将输出从Device拷回Host
    
    // 8. 后处理: 找出Top5索引，读取标签并打印结果
    
    // 9. 资源释放
    
    return 0;
}   
```


请同学们仔细阅读上述代码，填充缺失的细节（如Top-5排序、标签打印等）。特别关注：
- 内存管理接口：`aclrtMalloc`/`aclrtFree`、`aclrtMemcpy`。
- 模型加载：`aclmdlLoadFromFileWithMem` 使用预分配内存可控制复用。
- 数据集创建：`aclmdlCreateDataset`、`aclCreateDataBuffer`。
- 执行与同步：`aclmdlExecute`、`aclrtSynchronizeStream`。

#### 步骤2：编译与运行
在 `src/` 目录下创建 `Makefile` 或直接使用g++编译：
```bash
cd /home/developer/experiment/lab_06/src
export ASCEND_HOME=/opt/home/developer/Ascend/cann-8.5.2

g++ main.c -o main -I$ASCEND_HOME/include -L$ASCEND_HOME/lib64 -lascendcl -ldl -lpthread
```
若编译成功，运行：
```bash
./main
```
观察输出结果，确认程序能正确执行并打印Top-5分类标签。若出现错误，按ACL返回码排查。

#### 步骤3：理解流程控制
修改代码，尝试完成以下练习：
- 不预先指定模型输入大小，通过`aclmdlGetInputSizeByIndex`动态获取。
- 使用两个Stream异步执行，并观察同步差异。
- 添加性能计时（`aclrtGetRunTime`或系统时间），统计推理耗时。


## 六、实验总结
通过本次实验，我们：
- 掌握了AscendCL编程的基本流程，能够独立完成设备初始化、内存管理、模型加载与执行、数据搬运等关键环节。
- 熟悉了`aclrtMalloc`、`aclrtMemcpy`等内存管理API，区分了Host与Device内存，并通过显式拷贝保证数据一致性。
- 理解了离线模型在ACL中的加载方式，掌握了模型描述信息获取与输入输出Dataset的构建方法。
- 初步实践了Ascend C算子的编写与集成，建立了从算子开发到应用调用的全栈思维。
- 通过代码调试与性能测试，锻炼了分析问题和优化程序的能力，为后续基于昇腾的AI应用开发奠定了坚实基础。

建议课后进一步阅读CANN开发文档，探索更复杂的模型（如YOLO、BERT）和多Device协同场景，巩固ACL编程技能。